# TASK 1. PROJECT OVERVIEW & KEY LEARNING OBJECTIVES

This is the **Gemini version** of the original OpenAI "Autonomous AI Agents with Guardrails and Handoffs" notebook.  
Every cell is a line-by-line equivalent using **Google's free Gemini model**.

### Key Concepts Implemented
- **Guardrails** — Input validation (e.g., blocking political topics) before the agent processes a request
- **Agents-as-Tools** — One agent can call another agent as a tool (Writer calls Searcher, Fundamentals)
- **Handoffs** — One agent completes its work and passes control to the next agent (Planner → Writer)

### OpenAI → Gemini Mapping

| OpenAI Agents SDK | Gemini Equivalent |
|---|---|
| `Agent(name, instructions, model, tools, output_type)` | `genai.GenerativeModel(model_name, tools, system_instruction, generation_config)` |
| `@function_tool` | Plain Python function |
| `@input_guardrail` decorator | Custom `run_with_guardrail()` function |
| `agent.as_tool(...)` | Custom wrapper that runs sub-agent and returns result |
| `handoff(agent, input_type, on_handoff)` | Custom `handoff_planner_to_writer()` function |
| `Runner.run(agent, input, session)` | `run_agent()` helper with function-call loop |
| `SQLiteSession` | `conversation_history` list |
| `model="gpt-4.1-mini"` | `"models/gemini-2.0-flash"` (free) |
| `model="gpt-4.1-nano"` | `"models/gemini-2.0-flash"` (free, same model) |

# TASK 2. SETUP API KEYS & TOOLS

In [1]:
# ── Original Cell 3: pip install ─────────────────────────────────────────────
# ORIGINAL:  %pip install -q --upgrade openai-agents==0.2.2
#            %pip install helper
# GEMINI:    Replace openai-agents with google-generativeai

%pip install -q google-generativeai python-dotenv requests pydantic

Note: you may need to restart the kernel to use updated packages.


In [9]:
# ── Original Cell 4: Imports & API keys ──────────────────────────────────────
# ORIGINAL:
#   import os, requests, asyncio
#   from agents import Agent, Runner, function_tool, SQLiteSession
#   from agents import handoff, RunContextWrapper, CodeInterpreterTool,
#                      input_guardrail, GuardrailFunctionOutput, TResponseInputItem
#   from agents.extensions import handoff_filters
#   from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
#   openai_api_key = os.getenv("OPENAI_API_KEY")
#
# GEMINI EQUIVALENT:
#   We don't need ANY of the agents SDK imports.
#   We build guardrails, handoffs, and agent-as-tool patterns ourselves.

import os
import json
import requests
from datetime import datetime
from IPython.display import display, Markdown
from dotenv import load_dotenv
from pydantic import BaseModel
from typing_extensions import TypedDict
import google.generativeai as genai

load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

genai.configure(api_key=gemini_api_key)

print("✅ API keys loaded")
print(f"Gemini: {gemini_api_key[:5]}***")
print(f"Tavily: {tavily_api_key[:5]}***")

✅ API keys loaded
Gemini: AIzaS***
Tavily: tvly-***


In [10]:
# ── Original Cell 5: print_markdown helper ───────────────────────────────────
# IDENTICAL — nothing model-specific

def print_markdown(txt: str):
    display(Markdown(txt))

In [16]:
# ── Original Cell 6: Define models ───────────────────────────────────────────
# ORIGINAL:
#   main_model = "gpt-4.1-mini"
#   small_model = "gpt-4.1-nano"
#
# GEMINI EQUIVALENT:
#   The legacy google-generativeai package requires "models/" prefix.

main_model = "models/gemini-3.1-flash-lite-preview"     # Replaces gpt-4.1-mini (free)
small_model = "models/gemini-3.1-flash-lite-preview"    # Replaces gpt-4.1-nano (same free model)

print(f"Main model:  {main_model}")
print(f"Small model: {small_model}")

Main model:  models/gemini-3.1-flash-lite-preview
Small model: models/gemini-3.1-flash-lite-preview


In [17]:
# ── Original Cell 7: Tavily search tool ──────────────────────────────────────
# ORIGINAL:
#   class TavilyParams(TypedDict): ...
#   @function_tool
#   def tavily_search(params: TavilyParams) -> str:
#
# GEMINI EQUIVALENT:
#   No decorator. Direct keyword args instead of TypedDict dict.

class TavilyParams(TypedDict):
    query: str
    max_results: int


def tavily_search(query: str, max_results: int = 3) -> str:
    """Return a newline-joined summary of Tavily search results.

    Args:
        query: The search query string.
        max_results: Maximum number of results to return.

    Returns:
        A formatted string of search results.
    """
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": tavily_api_key,
        "query": query,
        "max_results": max_results,
    }
    resp = requests.post(url, json=payload, headers={"Content-Type": "application/json"})
    if resp.status_code != 200:
        return f"Tavily error {resp.status_code}"
    items = resp.json().get("results", [])
    return "\n".join([f"- {itm['title']}: {itm['content']}" for itm in items]) or "No hits"


print("✅ Tavily search tool ready.")

✅ Tavily search tool ready.


In [18]:
# ── CORE HELPER: run_agent — Replaces Runner.run() ───────────────────────────
# Handles the Gemini function-calling loop automatically.
# Works for models WITH tools (tavily_search) and WITHOUT tools.

def run_agent(model, user_input: str, history: list = None) -> str:
    """
    Runs a Gemini agent with automatic function-calling loop.
    Equivalent to: await Runner.run(agent, input, session)
    """
    if history is None:
        history = []

    chat = model.start_chat(history=history)
    response = chat.send_message(user_input)

    max_iterations = 10
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        parts = response.candidates[0].content.parts

        # Find function call if any
        function_call_part = None
        for part in parts:
            if hasattr(part, "function_call") and part.function_call.name:
                function_call_part = part
                break

        if function_call_part:
            fc = function_call_part.function_call
            fn_name = fc.name
            fn_args = dict(fc.args)
            print(f"  🔧 Tool call: {fn_name}({fn_args})")

            # Execute the function
            if fn_name == "tavily_search":
                result = tavily_search(**fn_args)
            else:
                result = f"Unknown function: {fn_name}"

            # Send result back
            response = chat.send_message(
                genai.protos.Content(
                    parts=[genai.protos.Part(
                        function_response=genai.protos.FunctionResponse(
                            name=fn_name, response={"result": result}
                        )
                    )]
                )
            )
        else:
            break

    # Extract text
    final_text = ""
    for part in response.candidates[0].content.parts:
        if hasattr(part, "text") and part.text:
            final_text += part.text

    history.extend(chat.history)
    return final_text


print("✅ run_agent() helper defined (replaces Runner.run).")

✅ run_agent() helper defined (replaces Runner.run).


# TASK 3. DEFINE AN AI AGENT (PLANNER) WITH GUARDRAIL

In [19]:
# ── Original Cell 9: Political topic guardrail ───────────────────────────────
# ORIGINAL:
#   class PoliticalTopicOutput(BaseModel):
#       is_political: bool
#       reasoning: str
#
#   politics_guardrail_agent = Agent(
#       name="Guardrail check",
#       instructions="Check if the user is asking about political topics...",
#       output_type=PoliticalTopicOutput)
#
#   @input_guardrail
#   async def politics_guardrail(ctx, agent, input):
#       result = await Runner.run(politics_guardrail_agent, input, context=ctx.context)
#       return GuardrailFunctionOutput(
#           output_info=result.final_output,
#           tripwire_triggered=result.final_output.is_political)
#
# GEMINI EQUIVALENT:
#   1. Create a Gemini model that returns JSON {is_political, reasoning}
#   2. Write a plain function that calls it and returns True/False
#   No decorators or SDK classes needed.

# Pydantic model (same as original)
class PoliticalTopicOutput(BaseModel):
    is_political: bool
    reasoning: str


# Guardrail agent — returns structured JSON
politics_guardrail_agent = genai.GenerativeModel(
    model_name=small_model,
    system_instruction="""Check if the user is asking about political topics, politicians, elections, 
government policy, or anything related to politics. 
Return a JSON object with:
- "is_political": true or false
- "reasoning": explanation of why""",
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        response_schema={
            "type": "object",
            "properties": {
                "is_political": {"type": "boolean"},
                "reasoning": {"type": "string"}
            },
            "required": ["is_political", "reasoning"]
        }
    ),
)


# Guardrail function — replaces @input_guardrail decorator
def check_politics_guardrail(user_input: str) -> PoliticalTopicOutput:
    """
    Checks if user input is about political topics.
    Returns PoliticalTopicOutput with is_political flag.
    
    Replaces:
        @input_guardrail
        async def politics_guardrail(ctx, agent, input) -> GuardrailFunctionOutput
    """
    raw = run_agent(politics_guardrail_agent, user_input)
    parsed = PoliticalTopicOutput(**json.loads(raw))
    return parsed


print("✅ Political topic guardrail ready.")

✅ Political topic guardrail ready.


In [20]:
check_politics_guardrail("What is the current stock price of Apple Inc.?")

PoliticalTopicOutput(is_political=False, reasoning="The user is asking for financial information regarding a publicly traded company's stock price, which is a business and economic topic, not a political one.")

In [21]:
# ── Original Cell 10: Planner Agent ──────────────────────────────────────────
# ORIGINAL:
#   class SearchPlanItem(BaseModel): ...
#   class SearchPlan(BaseModel): ...
#   planner_agent = Agent(
#       name="Planner", instructions=..., model=main_model,
#       output_type=SearchPlan, input_guardrails=[politics_guardrail])
#
# GEMINI EQUIVALENT:
#   - Agent → genai.GenerativeModel with JSON output schema
#   - input_guardrails → handled by run_with_guardrail() wrapper below

class SearchPlanItem(BaseModel):
    reason: str
    query: str

class SearchPlan(BaseModel):
    searches: list[SearchPlanItem]


date = datetime.now().strftime("%Y-%m-%d")

planner_agent = genai.GenerativeModel(
    model_name=main_model,
    system_instruction=f"""Current date: {date}
Context: You are a research planner agent tasked with designing a comprehensive research plan.
You have access to web search tools and should utilize the current date ({date}) when planning.
Instruction: Break down the user's request into 3 distinct web searches, each with a clear reason and a specific query.
Ensure coverage of recent news, company fundamentals, risks, sentiment, and broader context.
Input: The user's research request and the current date.
Output: A JSON object with a "searches" array, each item having "reason" and "query" fields.""",
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        response_schema={
            "type": "object",
            "properties": {
                "searches": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "reason": {"type": "string"},
                            "query": {"type": "string"}
                        },
                        "required": ["reason", "query"]
                    }
                }
            },
            "required": ["searches"]
        }
    ),
)


# Wrapper that runs guardrail BEFORE the planner
# Replaces: input_guardrails=[politics_guardrail] in Agent()
def run_planner_with_guardrail(user_input: str) -> SearchPlan:
    """
    Runs the politics guardrail check first, then the planner agent.
    Raises an exception if the guardrail is triggered.
    
    Replaces:
        await Runner.run(starting_agent=planner_agent, input=q1)
        where planner_agent had input_guardrails=[politics_guardrail]
    """
    # Step 1: Run guardrail
    print("🛡️ Running politics guardrail check...")
    guardrail_result = check_politics_guardrail(user_input)
    
    if guardrail_result.is_political:
        raise ValueError(
            f"❌ GUARDRAIL TRIGGERED: Political topic detected!\n"
            f"   Reasoning: {guardrail_result.reasoning}"
        )
    
    print(f"  ✅ Guardrail passed. Reasoning: {guardrail_result.reasoning}")
    
    # Step 2: Run planner
    print("📋 Running Planner agent...")
    raw = run_agent(planner_agent, user_input)
    plan = SearchPlan(**json.loads(raw))
    return plan


print(f"✅ Planner agent ready (with guardrail). Date: {date}")

✅ Planner agent ready (with guardrail). Date: 2026-04-03


In [22]:
# ── Original Cell 11: Test the planner with guardrail ────────────────────────
# ORIGINAL:
#   q1 = "solid state battery companies"
#   run1 = await Runner.run(starting_agent=planner_agent, input=q1)
#   print_markdown(f"### 🤖 Agent's Answer\n{run1.final_output}")
#
# GEMINI EQUIVALENT:

# Test 1: Non-political topic (should PASS guardrail)
q1 = "solid state battery companies"
try:
    plan1 = run_planner_with_guardrail(q1)
    print_markdown(f"### 🤖 Agent's Answer\n```json\n{plan1.model_dump_json(indent=2)}\n```")
except ValueError as e:
    print_markdown(f"### 🚫 Blocked\n{e}")

print("\n" + "="*60 + "\n")

# Test 2: Political topic (should TRIGGER guardrail)
q2 = "Why is Trump meeting with Putin this week?"
try:
    plan2 = run_planner_with_guardrail(q2)
    print_markdown(f"### 🤖 Agent's Answer\n```json\n{plan2.model_dump_json(indent=2)}\n```")
except ValueError as e:
    print_markdown(f"### 🚫 Blocked\n{e}")

🛡️ Running politics guardrail check...
  ✅ Guardrail passed. Reasoning: The query asks for information regarding technology companies in the energy sector, which is a business and technical topic, not a political one.
📋 Running Planner agent...


### 🤖 Agent's Answer
```json
{
  "searches": [
    {
      "reason": "Identify key public and private companies currently leading the development and commercialization of solid-state battery technology as of early 2026.",
      "query": "top solid state battery companies to watch 2026"
    },
    {
      "reason": "Assess the fundamental business progress, recent manufacturing milestones, and major automotive partnerships for industry leaders like QuantumScape and Solid Power.",
      "query": "QuantumScape Solid Power battery development progress news April 2026"
    },
    {
      "reason": "Understand the current market sentiment, technical hurdles, and competitive risks facing the solid-state battery industry including production scalability and cost challenges.",
      "query": "challenges and market outlook for solid state batteries 2026"
    }
  ]
}
```



🛡️ Running politics guardrail check...


### 🚫 Blocked
❌ GUARDRAIL TRIGGERED: Political topic detected!
   Reasoning: The query concerns the actions of a former U.S. President and a foreign head of state, which falls under the category of international relations and political events.

**PRACTICE OPPORTUNITY:**
- **Create a new guardrail that will block the agent from discussing topics related to defence and military.**
  - **Update the guardrail agent's instructions so that it focuses on analyzing defence and military-related issues and risks.**
  - **Instantiate the `defense_guardrail` agent.**
  - **Provide an example input (e.g., "Lockheed Martin drones") and test the agent.**

# TASK 4. DEFINE A TEAM OF AI AGENTS: SEARCH & FUNDAMENTAL ANALYSIS AGENTS

In [23]:
# ── Original Cell 15: Search Agent ───────────────────────────────────────────
# ORIGINAL:
#   class Summary(BaseModel): summary: str
#   search_agent = Agent(name="Searcher", instructions=...,
#       tools=[tavily_search], model=main_model, output_type=Summary)
#
# GEMINI EQUIVALENT:

class Summary(BaseModel):
    summary: str

search_agent = genai.GenerativeModel(
    model_name=main_model,
    tools=[tavily_search],
    system_instruction="""Context: You are a search specialist agent with access to the tavily_search tool.
Your goal is to provide up-to-date, relevant information for a research task.
Instruction: Use tavily_search to find the most recent and pertinent information related to the user's query.
Summarize your findings clearly and concisely in no more than 200 words.
Input: The user's search query.
Output: A concise summary (200 words max) of the most relevant and recent information found.""",
)

print("✅ Search agent ready.")

✅ Search agent ready.


In [24]:
# ── Original Cell 16: Fundamentals Analysis Agent ────────────────────────────
# ORIGINAL:
#   fundamentals_agent = Agent(name="FundamentalsAnalyst", instructions=...,
#       output_type=Summary, model=main_model, tools=[tavily_search])
#
# GEMINI EQUIVALENT:

fundamentals_agent = genai.GenerativeModel(
    model_name=main_model,
    tools=[tavily_search],
    system_instruction="""Context: You are a financial analyst specializing in company fundamentals.
Instruction: Carefully analyze the provided notes to assess the company's financial fundamentals,
including revenue, growth, and margins. Use tavily_search to find additional data if needed.
Input: Notes containing relevant financial data and qualitative information about the company.
Output: A concise summary (200 words max) highlighting key points about the company's revenue,
growth trajectory, and profit margins.""",
)

print("✅ Fundamentals agent ready.")

✅ Fundamentals agent ready.


# TASK 5: CREATE AGENTS AS TOOLS FOR THE WRITER AGENT

In [25]:
# ── Original Cell 18: Writer Agent with agents-as-tools ──────────────────────
# ORIGINAL:
#   class FinalReport(BaseModel): ...
#   async def extract_summary(run_result: RunResult) -> str: ...
#   writer_agent = Agent(name="Writer", ..., tools=[
#       fundamentals_agent.as_tool("fundamentals", ..., custom_output_extractor=extract_summary),
#       search_agent.as_tool("search", ..., custom_output_extractor=extract_summary),
#   ])
#
# GEMINI EQUIVALENT:
#   Gemini doesn't have agent.as_tool(). Instead we:
#   1. Define wrapper functions that call sub-agents and return their output
#   2. Pass these wrapper functions as tools to the writer model

class FinalReport(BaseModel):
    short_summary: str
    markdown_report: str
    follow_up_questions: list[str]


# ── Agent-as-tool wrappers ────────────────────────────────────────────────────
# These replace: fundamentals_agent.as_tool(...) and search_agent.as_tool(...)

def search_tool(query: str) -> str:
    """Search the web for the most recent information about a company or topic.
    This tool uses a search specialist agent to find and summarize relevant results.

    Args:
        query: The search query about a company or topic.

    Returns:
        A concise summary of the most relevant search results.
    """
    print(f"  📡 [Search Agent] Searching for: {query}")
    result = run_agent(search_agent, query)
    print(f"  ✅ [Search Agent] Done.")
    return result


def fundamentals_tool(query: str) -> str:
    """Analyze the financial fundamentals of a company including revenue, growth, and margins.
    This tool uses a financial analyst agent.

    Args:
        query: The company name or topic to analyze fundamentals for.

    Returns:
        A concise summary of the company's financial fundamentals.
    """
    print(f"  📊 [Fundamentals Agent] Analyzing: {query}")
    result = run_agent(fundamentals_agent, query)
    print(f"  ✅ [Fundamentals Agent] Done.")
    return result


# ── Writer agent (uses sub-agents as tools) ──────────────────────────────────
# IMPORTANT: Gemini does NOT allow combining tools (function calling) with
# response_mime_type='application/json'. You get:
#   InvalidArgument: 400 Function calling with a response mime type: 'application/json' is unsupported
# FIX: Remove generation_config and enforce JSON output via the system prompt instead.

writer_agent = genai.GenerativeModel(
    model_name=main_model,
    tools=[search_tool, fundamentals_tool],     # Sub-agents wrapped as callable tools
    system_instruction="""Context: You are an expert research writer preparing a comprehensive investment report.
Your audience is sophisticated and expects clarity, depth, and actionable insights.

Instruction: Synthesize information into a cohesive, well-structured markdown report of at least 600 words.
Your report must:
(1) Begin with a concise 2-3 sentence executive summary
(2) Integrate and cross-reference key facts, trends, and perspectives
(3) Organize content with clear headings and logical flow
(4) Maintain objectivity, cite evidence, and avoid speculation
(5) Conclude with 3-5 insightful follow-up research questions

You MUST always use the 'search_tool' to gather up-to-date information.
The 'fundamentals_tool' is optional — only use it if the user requests fundamentals analysis.

CRITICAL OUTPUT FORMAT: After using tools and gathering all information, your FINAL response
must be ONLY a valid JSON object (no markdown fences, no extra text) with exactly these fields:
{
  "short_summary": "2-3 sentence executive summary",
  "markdown_report": "detailed markdown report (600+ words)",
  "follow_up_questions": ["question 1", "question 2", "question 3"]
}
Do NOT wrap the JSON in ```json``` code fences. Return ONLY the raw JSON object.""",
    # NOTE: No generation_config here — cannot combine tools + response_mime_type
)

print("✅ Writer agent ready (with search and fundamentals sub-agents as tools).")

✅ Writer agent ready (with search and fundamentals sub-agents as tools).


In [26]:
# ── Enhanced run_agent for Writer (handles sub-agent tool calls) ──────────────
# The writer calls search_tool and fundamentals_tool which are regular functions.
# We need a run_agent variant that can dispatch to ANY registered tool function.

def run_writer_agent(model, user_input: str, history: list = None) -> str:
    """
    Runs the writer agent, dispatching tool calls to the correct function.
    Handles search_tool, fundamentals_tool, and tavily_search.
    """
    if history is None:
        history = []

    # Map of tool name → function
    tool_map = {
        "search_tool": search_tool,
        "fundamentals_tool": fundamentals_tool,
        "tavily_search": tavily_search,
    }

    chat = model.start_chat(history=history)
    response = chat.send_message(user_input)

    max_iterations = 15
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        parts = response.candidates[0].content.parts

        function_call_part = None
        for part in parts:
            if hasattr(part, "function_call") and part.function_call.name:
                function_call_part = part
                break

        if function_call_part:
            fc = function_call_part.function_call
            fn_name = fc.name
            fn_args = dict(fc.args)
            print(f"  🔧 Writer calling: {fn_name}({fn_args})")

            fn = tool_map.get(fn_name)
            if fn:
                result = fn(**fn_args)
            else:
                result = f"Unknown function: {fn_name}"

            response = chat.send_message(
                genai.protos.Content(
                    parts=[genai.protos.Part(
                        function_response=genai.protos.FunctionResponse(
                            name=fn_name, response={"result": result}
                        )
                    )]
                )
            )
        else:
            break

    final_text = ""
    for part in response.candidates[0].content.parts:
        if hasattr(part, "text") and part.text:
            final_text += part.text

    history.extend(chat.history)
    return final_text


print("✅ run_writer_agent() defined.")

✅ run_writer_agent() defined.


In [27]:
# ── Original Cell 19: Test the Writer Agent ──────────────────────────────────
# ORIGINAL:
#   q1 = "Do a deep dive on the latest news in Hindustan Lever stock..."
#   run1 = await Runner.run(starting_agent=writer_agent, input=q1)
#   print_markdown(f"### 🤖 Agent's Answer\n{run1.final_output}")
#
# GEMINI EQUIVALENT:

q1 = "Do a deep dive on the latest news in Hindustan Lever stock. I also need the fundamental analysis of the company"

print_markdown(f"**User:** {q1}")
raw_output = run_writer_agent(writer_agent, q1)

# Clean JSON — Gemini sometimes wraps output in ```json ... ``` fences
import re
cleaned = re.sub(r'^```(?:json)?\s*', '', raw_output.strip())
cleaned = re.sub(r'\s*```$', '', cleaned.strip())

try:
    report = FinalReport(**json.loads(cleaned))
    print_markdown(f"### 📝 Executive Summary\n{report.short_summary}")
    print_markdown("---")
    print_markdown(f"### 📄 Full Report\n{report.markdown_report}")
    print_markdown("---")
    print_markdown("### 🔍 Follow-Up Questions\n- " + "\n- ".join(report.follow_up_questions))
except Exception as e:
    print(f"⚠️ Parse error: {e}")
    print_markdown(raw_output)

**User:** Do a deep dive on the latest news in Hindustan Lever stock. I also need the fundamental analysis of the company

### 📝 Executive Summary
Hindustan Unilever Limited (HUL) is currently navigating a complex landscape characterized by resilient premium segment growth tempered by sluggish rural demand and heightened competitive intensity. While the company maintains its long-term structural advantage, recent quarterly results underscore the ongoing challenge of balancing volume recovery with margin protection amid fluctuating commodity costs.

---

### 📄 Full Report
# Investment Analysis: Hindustan Unilever Limited (HUL)

## Executive Overview
Hindustan Unilever Limited (HUL) remains the bellwether of the Indian Fast-Moving Consumer Goods (FMCG) sector. As India's largest consumer goods company, its performance is often viewed as a proxy for the broader Indian rural and urban consumption story. Recent developments highlight a company at a strategic inflection point, managing the duality of strong urban premiumization against persistent rural volume stagnation.

## Recent News and Market Developments
In recent quarters, HUL’s performance has reflected the broader macroeconomic pressures impacting the Indian FMCG sector. The company's recent earnings reports have consistently highlighted a divergence in performance between its premium and mass-market portfolios. 

Key areas of recent focus include:

1.  **Rural vs. Urban Dynamics:** While urban consumption has shown resilience—driven by rising disposable incomes and premiumization trends—rural markets have been slower to recover. Inflationary pressures on essential food items have historically dampened rural discretionary spending. However, recent data suggests a moderate recovery in rural demand, supported by favorable monsoon expectations and government spending.

2.  **Competitive Intensity:** HUL is currently facing increased competition from regional and local players who are capitalizing on the softening of raw material prices (such as palm oil and crude derivatives) to lower their price points. This has forced HUL to invest more heavily in advertising and promotions to protect its market share, impacting short-term margins.

3.  **Strategic Portfolio Review:** HUL has been actively rebalancing its portfolio. Recent discussions and investor communications indicate a strong focus on high-growth segments like beauty, personal care, and health foods. The company is leaning heavily into its 'Winnable Categories'—those where it has a distinct competitive advantage and high growth potential.

## Fundamental Analysis
HUL continues to exhibit a robust balance sheet and superior capital efficiency, hallmark traits that have historically justified its premium valuation multiples.

*   **Revenue Growth and Market Penetration:** HUL’s vast distribution network—reaching millions of outlets—remains its most formidable competitive moat. Revenue growth is increasingly driven by volume-led growth rather than just price hikes, signaling a healthier underlying demand environment as inflationary pressures stabilize.

*   **Margins and Profitability:** The company has demonstrated strong margin resilience through a combination of 'Net Revenue Management' (NRM) and stringent cost-saving programs (e.g., 'Project Symphony'). While gross margins have seen fluctuations due to volatile input costs, HUL has consistently managed its EBITDA margins within a tight band, showcasing exceptional operational discipline.

*   **Return Metrics:** HUL maintains best-in-class Return on Capital Employed (ROCE) and Return on Equity (ROE). Its asset-light model in several categories and high cash conversion cycle allow it to consistently reward shareholders through dividends while reinvesting in brand equity.

## Strategic Challenges and Outlook
The primary challenge for HUL remains the speed of rural recovery. As the company scales its digital transformation efforts (e.g., Shikhar app for B2B ordering), it is better positioned than ever to gather real-time data on consumer behavior. However, the macro-environment remains unpredictable. 

Investors should monitor the company’s ability to defend its market share against mid-tier regional players without sacrificing its long-term brand equity or profitability. The pivot toward premiumization is a double-edged sword: it drives higher margins but leaves the mass-market flank vulnerable to aggressive local discounting. HUL’s ability to segment its portfolio effectively to capture both value-conscious and premium consumers will be the primary driver of its stock performance over the next 24 months.

In conclusion, HUL remains a defensive pillar in an Indian equity portfolio, offering stability and steady compounding. While short-term volatility persists due to competitive pressures, the long-term structural tailwinds of India’s demographic dividend and rising consumption power remain firmly intact.

---

### 🔍 Follow-Up Questions
- How is HUL specifically adjusting its marketing strategy to counter the resurgence of regional 'value' brands in the mass-market segment?
- What impact do potential fluctuations in global palm oil and crude oil prices have on HUL's operating margins for the upcoming two quarters?
- To what extent has the adoption of the 'Shikhar' digital distribution platform contributed to market share gains in non-metro geographies?
- How does HUL's current valuation multiple compare to its 5-year historical average, and what does this imply for expected returns for new investors?

**PRACTICE OPPORTUNITY:**
- **Review the output above to see which tools were used and how many times.**
- **The print statements show the trace hierarchy: Writer → Search Agent → tavily_search, Writer → Fundamentals Agent → tavily_search**

# TASK 6. AI AGENTS HANDOFF

In [28]:
# ── Original Cell 23: Session ────────────────────────────────────────────────
# ORIGINAL: session = SQLiteSession("research_agent_handoff")
# GEMINI:   Simple list-based history

session_history = []  # Replaces SQLiteSession
print("✅ Session history initialized.")

✅ Session history initialized.


In [29]:
# ── Original Cell 24 & 25: Handoff from Planner → Writer ────────────────────
# ORIGINAL:
#   class PlannerToWriterInput(BaseModel): ...
#   def on_planner_to_writer(ctx, input_data): ...
#   handoff_to_writer = handoff(agent=writer_agent, ...)
#   planner_with_handoff = planner_agent.clone(handoffs=[handoff_to_writer])
#
#   async def run_handoffs_demo(user_query):
#       run_res = await Runner.run(planner_with_handoff, user_query, session=session)
#       report = run_res.final_output
#       ... display report ...
#
# GEMINI EQUIVALENT:
#   We implement handoff as a simple function that:
#   1. Runs the Planner (with guardrail) to get a search plan
#   2. Prints the handoff message
#   3. Passes the plan + original query to the Writer agent
#   4. Displays the final report

class PlannerToWriterInput(BaseModel):
    original_query: str
    search_plan: SearchPlan


def run_handoffs_demo(user_query: str):
    """
    Full handoff pipeline: Planner (with guardrail) → Writer
    Replaces the original async run_handoffs_demo with:
        planner_with_handoff → handoff_to_writer → writer_agent
    """
    print_markdown(f"## 🕵️ User Query\n{user_query}")

    # ── Phase 1: Planner (with guardrail) ────────────────────────────────────
    try:
        search_plan = run_planner_with_guardrail(user_query)
    except ValueError as e:
        print_markdown(f"### 🚫 Blocked by Guardrail\n{e}")
        return None

    print_markdown(f"### 📋 Search Plan\n```json\n{search_plan.model_dump_json(indent=2)}\n```")

    # ── Handoff: Planner → Writer ────────────────────────────────────────────
    print_markdown("➡️ **Transfer: Planner → Writer**")

    # Build the input for the writer, combining original query + search plan
    writer_input = (
        f"Original query: {user_query}\n\n"
        f"Search plan from Planner agent:\n{search_plan.model_dump_json(indent=2)}\n\n"
        f"Please execute the search plan using the search_tool for each query, "
        f"and if fundamentals analysis is requested, use fundamentals_tool as well. "
        f"Then synthesize everything into a comprehensive investment report."
    )

    # ── Phase 2: Writer ───────────────────────────────────────────────────────
    print("\n✍️ Running Writer agent...")
    raw_output = run_writer_agent(writer_agent, writer_input, session_history)
    print("  ✅ Writer done.\n")

    # Clean JSON — Gemini sometimes wraps output in ```json ... ``` fences
    import re
    cleaned = re.sub(r'^```(?:json)?\s*', '', raw_output.strip())
    cleaned = re.sub(r'\s*```$', '', cleaned.strip())

    # Parse and display
    try:
        report = FinalReport(**json.loads(cleaned))
    except Exception as e:
        print(f"⚠️ Parse error: {e}")
        print_markdown(raw_output)
        return None

    print_markdown("---")
    print_markdown(f"### 🔎 Executive Summary\n{report.short_summary}")
    print_markdown("\n\n-----------------\n\n")
    print_markdown(f"### 📄 Full Report\n{report.markdown_report}")
    print_markdown("\n\n-----------------\n\n")
    print_markdown(
        "### 🔍 Follow-Up Questions\n- "
        + "\n- ".join(report.follow_up_questions)
    )
    print_markdown("\n\n-----------------\n\n")

    return report


print("✅ run_handoffs_demo() defined (Planner → Writer pipeline).")

✅ run_handoffs_demo() defined (Planner → Writer pipeline).


In [30]:
# ── Original Cell 26: Test handoff with political query (should be blocked) ──
# ORIGINAL:
#   handoff_result = await run_handoffs_demo(
#       "Can you let me know about the stock holdings of people who are in congress")
#
# GEMINI EQUIVALENT:

handoff_result = run_handoffs_demo(
    "Can you let me know about the stock holdings of people who are in congress"
)

## 🕵️ User Query
Can you let me know about the stock holdings of people who are in congress

🛡️ Running politics guardrail check...


### 🚫 Blocked by Guardrail
❌ GUARDRAIL TRIGGERED: Political topic detected!
   Reasoning: The request specifically asks about the stock holdings of members of Congress, which relates to government officials and potential conflicts of interest within the political sphere.

In [31]:
# ── Original Cell 27: Test handoff with stock analysis query ─────────────────
# ORIGINAL:
#   handoff_result = await run_handoffs_demo(
#       "Do a deep dive on AAPL stock. Also I need the fundamentals analysis.")
#
# GEMINI EQUIVALENT:

session_history = []  # Fresh session

handoff_result = run_handoffs_demo(
    "Do a deep dive on the latest news and developments in the AAPL stock. Also I need the fundamentals analysis of the company."
)

## 🕵️ User Query
Do a deep dive on the latest news and developments in the AAPL stock. Also I need the fundamentals analysis of the company.

🛡️ Running politics guardrail check...
  ✅ Guardrail passed. Reasoning: The request pertains to financial analysis of Apple Inc. (AAPL) stock, which is a business and market-related topic, not a political one.
📋 Running Planner agent...


### 📋 Search Plan
```json
{
  "searches": [
    {
      "reason": "Identify the latest news, product announcements, and market developments for Apple Inc. as of April 2026.",
      "query": "Apple Inc. AAPL latest news and developments April 2026"
    },
    {
      "reason": "Analyze Apple's current financial health, including recent earnings reports, revenue trends, and key valuation metrics.",
      "query": "Apple Inc. AAPL fundamental analysis Q1 2026 earnings"
    },
    {
      "reason": "Evaluate potential risks and market sentiment impacting AAPL, including regulatory hurdles and competitive landscape.",
      "query": "current risks and market sentiment for Apple stock AAPL 2026"
    }
  ]
}
```

➡️ **Transfer: Planner → Writer**


✍️ Running Writer agent...
  🔧 Writer calling: search_tool({'query': 'Apple Inc. AAPL latest news and developments April 2026'})
  📡 [Search Agent] Searching for: Apple Inc. AAPL latest news and developments April 2026
  🔧 Tool call: tavily_search({'query': 'Apple Inc. AAPL news developments April 2026'})
  ✅ [Search Agent] Done.
  🔧 Writer calling: fundamentals_tool({'query': 'AAPL'})
  📊 [Fundamentals Agent] Analyzing: AAPL
  🔧 Tool call: tavily_search({'query': 'Apple latest financial results revenue growth profit margins summary 2024'})
  ✅ [Fundamentals Agent] Done.
  ✅ Writer done.



...



-----------------



### 📄 Full Report
# Investment Analysis Report: Apple Inc. (AAPL)

## Executive Overview
As of April 2026, Apple Inc. (AAPL) is operating within a transitionary phase, navigating the maturity of its core hardware business while aggressively pivoting toward generative AI integration. While the stock has experienced recent volatility—dipping roughly 12% from its late-2025 peak—the company maintains its characteristic fundamental strength, driven by high-margin Services and a highly loyal ecosystem. Success in the near-to-mid-term will depend on the market's perception of Apple’s AI capabilities and its ability to reignite growth in the Chinese market.

## Market Context and Recent Developments
Heading into the second quarter of 2026, Apple is confronting a confluence of macroeconomic and competitive pressures. The market is currently grappling with a valuation reset, as the stock’s trailing P/E ratio of 32x is viewed by some analysts as overly ambitious, given the decelerating growth rates in the core iPhone business.

### The AI Competitive Landscape
Artificial Intelligence remains the primary narrative defining investor sentiment. Apple is under significant pressure to prove that its "Apple Intelligence" initiatives can compete with the rapid advancements made by Alphabet (Google) and other hyperscalers. Investors are looking for tangible evidence that these AI features will drive significant hardware upgrade cycles during the transition from the iPhone 17 to the iPhone 18. Failure to deliver a compelling AI differentiation could continue to hamper the company's valuation multiple.

### Geopolitical and Macroeconomic Factors
Weakness in the Chinese market continues to be a persistent overhang on the stock. Given China's historical role as both a critical manufacturing hub and a substantial revenue driver, the ongoing slump in domestic consumer demand for premium handsets has weighed heavily on revenue forecasts. While Apple has made strides to diversify its supply chain, the dependency on the Chinese consumer remains a strategic vulnerability.

## Fundamental Analysis
Apple’s financial profile remains defined by high capital efficiency and a disciplined shift toward services revenue.

### Revenue and Margin Trends
Apple continues to exhibit a model of mature, stable growth. Annual revenue remains near $391 billion, characterized by slow top-line growth (approximately 2% year-over-year). However, the narrative is not one of stagnation, but of margin expansion. The Services segment, which commands significantly higher margins than hardware, now represents a larger slice of the total revenue pie. This shift has helped support gross margins exceeding 46%, allowing the company to sustain strong profitability even in quarters where hardware unit volumes are under pressure.

### Capital Allocation
Apple's ability to generate immense free cash flow continues to be its greatest defensive attribute. This liquidity enables the company to engage in consistent share buybacks and dividend distributions, providing a floor for the stock during periods of broader market correction. The company’s ability to sustain shareholder value, regardless of immediate hardware growth spikes, remains a key driver for institutional support.

## Risks and Outlook
The investment case for AAPL in 2026 rests on balancing the stability of its Services-led model against the cyclical risks inherent in its hardware dependence. 

1. **Regulatory Hurdles:** Ongoing antitrust litigation in multiple jurisdictions remains a persistent risk to the Services revenue model, specifically concerning App Store fees and ecosystem lock-in.
2. **Hardware Saturation:** With the smartphone market in many developed economies reaching saturation, Apple is increasingly reliant on users upgrading to higher-tier models rather than acquiring new users. The efficacy of future product launches in driving these upgrades is paramount.
3. **Valuation Sensitivity:** As Apple evolves into a more service-dependent, slower-growth company, the market may continue to debate the appropriateness of its high P/E ratio, potentially leading to further volatility in the near term.

In summary, while the current environment is marked by skepticism regarding the company's AI leadership and hardware growth, Apple remains a dominant financial force with unparalleled pricing power and ecosystem stickiness. The long-term thesis hinges on the company's ability to successfully integrate its AI roadmap into the consumer experience to justify its current premium valuation.



-----------------



### 🔍 Follow-Up Questions
- How does the projected growth rate of Apple’s Services revenue for the next 18 months compare to the expected decline in hardware segment margins?
- What specific regulatory milestones in the EU and US should investors monitor to assess the potential impact on App Store revenue?
- To what extent does Apple's current R&D spending on artificial intelligence represent a departure from historical spending patterns, and what does this imply for future operating margins?



-----------------



**PRACTICE OPPORTUNITY:**
- **Create a new AI agent named SentimentAgent that specializes in searching online to find out the sentiment regarding a company.**
  - **Update the agent's instructions so that it focuses on analyzing online sentiment (positive, negative, neutral) about the company.**
  - **Instantiate the SentimentAgent agent.**
  - **Provide an example input (e.g., "What is the current market sentiment about Tesla?").**
  - **Run the agent, analyze the response.**

# PRACTICE OPPORTUNITY SOLUTIONS

**PRACTICE OPPORTUNITY SOLUTION: Defense Guardrail**

In [ ]:
# ── Original Cell 34: Defense guardrail solution ──────────────────────────────
# ORIGINAL:
#   class DefenseTopicOutput(BaseModel): ...
#   defense_guardrail_agent = Agent(name="Guardrail check", ...)
#   @input_guardrail
#   async def defense_guardrail(...): ...
#   planner_agent_with_defense_guardrail = Agent(..., input_guardrails=[defense_guardrail])
#
# GEMINI EQUIVALENT:

class DefenseTopicOutput(BaseModel):
    is_defense: bool
    reasoning: str


defense_guardrail_agent = genai.GenerativeModel(
    model_name=small_model,
    system_instruction="""Check if the user is asking about defence topics, military equipment, 
weapons systems, or anything related to defense and military.
Return a JSON object with:
- "is_defense": true or false
- "reasoning": explanation of why""",
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        response_schema={
            "type": "object",
            "properties": {
                "is_defense": {"type": "boolean"},
                "reasoning": {"type": "string"}
            },
            "required": ["is_defense", "reasoning"]
        }
    ),
)


def check_defense_guardrail(user_input: str) -> DefenseTopicOutput:
    raw = run_agent(defense_guardrail_agent, user_input)
    return DefenseTopicOutput(**json.loads(raw))


def run_planner_with_defense_guardrail(user_input: str) -> SearchPlan:
    print("🛡️ Running defense guardrail check...")
    result = check_defense_guardrail(user_input)
    
    if result.is_defense:
        raise ValueError(
            f"❌ GUARDRAIL TRIGGERED: Defense/military topic detected!\n"
            f"   Reasoning: {result.reasoning}"
        )
    
    print(f"  ✅ Guardrail passed. Reasoning: {result.reasoning}")
    print("📋 Running Planner agent...")
    raw = run_agent(planner_agent, user_input)
    return SearchPlan(**json.loads(raw))


print("✅ Defense guardrail + planner ready.")

In [ ]:
# ── Original Cell 35: Test defense guardrail ─────────────────────────────────
# ORIGINAL:
#   q1 = "Lockheed martin drones"   # should trigger
#   q1 = "GPT-5 Model"              # should pass
#   run1 = await Runner.run(starting_agent=planner_agent, input=q1)
#
# GEMINI EQUIVALENT:

# Test 1: Defense topic (should be BLOCKED)
print("=== Test: Defense topic ===")
try:
    plan = run_planner_with_defense_guardrail("Lockheed Martin drones")
    print_markdown(f"### Plan\n```json\n{plan.model_dump_json(indent=2)}\n```")
except ValueError as e:
    print_markdown(f"### 🚫 Blocked\n{e}")

print("\n" + "="*60 + "\n")

# Test 2: Non-defense topic (should PASS)
print("=== Test: Non-defense topic ===")
try:
    plan = run_planner_with_defense_guardrail("GPT-5 Model")
    print_markdown(f"### Plan\n```json\n{plan.model_dump_json(indent=2)}\n```")
except ValueError as e:
    print_markdown(f"### 🚫 Blocked\n{e}")

**PRACTICE OPPORTUNITY SOLUTION: Sentiment Agent + Full Pipeline**

In [ ]:
# ── Original Cell 40: Sentiment Agent + updated Writer + full pipeline ────────
# ORIGINAL:
#   sentiment_agent = Agent(name="SentimentAnalyst", ...)
#   writer_agent = Agent(..., tools=[fundamentals.as_tool, search.as_tool, sentiment.as_tool])
#   handoff + planner_with_handoff
#   handoff_result = await run_handoffs_demo("AAPL + fundamentals + sentiment")
#
# GEMINI EQUIVALENT:

# ── Sentiment Agent ──────────────────────────────────────────────────────────
sentiment_agent = genai.GenerativeModel(
    model_name=main_model,
    tools=[tavily_search],
    system_instruction="""Context: You are a sentiment analyst specializing in evaluating online sentiment about companies.
Instruction: Analyze search results to determine the current sentiment (positive, negative, or neutral).
Consider recent news, social media, and analyst opinions.
Input: Notes containing relevant information and search results about the company.
Output: A concise summary (200 words max) highlighting the overall sentiment, supporting evidence, 
and any notable trends or shifts in sentiment.""",
)


# ── Sentiment tool wrapper ───────────────────────────────────────────────────
def sentiment_tool(query: str) -> str:
    """Analyze the current market sentiment about a company using online sources.

    Args:
        query: The company name or topic to analyze sentiment for.

    Returns:
        A concise summary of the overall market sentiment.
    """
    print(f"  💬 [Sentiment Agent] Analyzing: {query}")
    result = run_agent(sentiment_agent, query)
    print(f"  ✅ [Sentiment Agent] Done.")
    return result


# ── Updated Writer with all 3 tools ──────────────────────────────────────────
writer_agent_v2 = genai.GenerativeModel(
    model_name=main_model,
    tools=[search_tool, fundamentals_tool, sentiment_tool],
    system_instruction="""Context: You are an expert research writer preparing a comprehensive investment report.

Instruction: Synthesize into a cohesive, well-structured markdown report of at least 600 words.
Your report must:
(1) Begin with a concise 2-3 sentence executive summary
(2) Integrate key facts, trends, and perspectives from all sources
(3) Organize content with clear headings and logical flow
(4) Maintain objectivity and cite evidence
(5) Conclude with 3-5 follow-up research questions

Tools available:
- search_tool: Search for latest news (REQUIRED - always use this)
- fundamentals_tool: Financial fundamentals analysis (optional)
- sentiment_tool: Market sentiment analysis (optional)

Output: Return a JSON object with:
- "short_summary": 2-3 sentence executive summary
- "markdown_report": detailed markdown report (600+ words)
- "follow_up_questions": array of 3-5 follow-up questions

CRITICAL OUTPUT FORMAT: After using tools and gathering all information, your FINAL response
must be ONLY a valid JSON object (no markdown fences, no extra text) with exactly these fields:
{"short_summary": "...", "markdown_report": "...", "follow_up_questions": ["..."]}
Do NOT wrap the JSON in code fences. Return ONLY the raw JSON object.""",
    # NOTE: No generation_config — cannot combine tools + response_mime_type
)


# ── Updated run_writer_agent for v2 (with sentiment_tool) ────────────────────
def run_writer_agent_v2(model, user_input: str, history: list = None) -> str:
    if history is None:
        history = []

    tool_map = {
        "search_tool": search_tool,
        "fundamentals_tool": fundamentals_tool,
        "sentiment_tool": sentiment_tool,
        "tavily_search": tavily_search,
    }

    chat = model.start_chat(history=history)
    response = chat.send_message(user_input)

    max_iterations = 20
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        parts = response.candidates[0].content.parts

        function_call_part = None
        for part in parts:
            if hasattr(part, "function_call") and part.function_call.name:
                function_call_part = part
                break

        if function_call_part:
            fc = function_call_part.function_call
            fn_name = fc.name
            fn_args = dict(fc.args)
            print(f"  🔧 Writer calling: {fn_name}({fn_args})")

            fn = tool_map.get(fn_name)
            result = fn(**fn_args) if fn else f"Unknown function: {fn_name}"

            response = chat.send_message(
                genai.protos.Content(
                    parts=[genai.protos.Part(
                        function_response=genai.protos.FunctionResponse(
                            name=fn_name, response={"result": result}
                        )
                    )]
                )
            )
        else:
            break

    final_text = ""
    for part in response.candidates[0].content.parts:
        if hasattr(part, "text") and part.text:
            final_text += part.text

    history.extend(chat.history)
    return final_text


# ── Updated handoff demo with v2 writer ──────────────────────────────────────
def run_handoffs_demo_v2(user_query: str):
    print_markdown(f"## 🕵️ User Query\n{user_query}")

    try:
        search_plan = run_planner_with_guardrail(user_query)
    except ValueError as e:
        print_markdown(f"### 🚫 Blocked by Guardrail\n{e}")
        return None

    print_markdown("➡️ **Transfer: Planner → Writer (v2 with Sentiment)**")

    writer_input = (
        f"Original query: {user_query}\n\n"
        f"Search plan:\n{search_plan.model_dump_json(indent=2)}\n\n"
        f"Execute the plan using search_tool, fundamentals_tool, and sentiment_tool as needed."
    )

    print("\n✍️ Running Writer v2 agent...")
    raw_output = run_writer_agent_v2(writer_agent_v2, writer_input)

    try:
        import re
        cleaned = re.sub(r'^```(?:json)?\s*', '', raw_output.strip())
        cleaned = re.sub(r'\s*```$', '', cleaned.strip())
        report = FinalReport(**json.loads(cleaned))
    except Exception as e:
        print(f"⚠️ Parse error: {e}")
        print_markdown(raw_output)
        return None

    print_markdown("---")
    print_markdown(f"### 🔎 Executive Summary\n{report.short_summary}")
    print_markdown("\n-----------------\n")
    print_markdown(f"### 📄 Full Report\n{report.markdown_report}")
    print_markdown("\n-----------------\n")
    print_markdown("### 🔍 Follow-Up Questions\n- " + "\n- ".join(report.follow_up_questions))
    return report


print("✅ Full pipeline v2 ready (with Sentiment Agent).")

In [ ]:
# ── Run the full v2 pipeline ─────────────────────────────────────────────────
# ORIGINAL:
#   handoff_result = await run_handoffs_demo(
#       "AAPL stock + fundamentals + sentiment analysis")
#
# GEMINI EQUIVALENT:

handoff_result = run_handoffs_demo_v2(
    "Do a deep dive on the latest news and developments in the AAPL stock. "
    "Also, I need the fundamentals and sentiment analysis of the company."
)